# LangChain messages: HumanMessage, AIMessage, ToolMessage

This notebook loads environment variables from `.env` and demonstrates the core LangChain message classes with three small examples.

In [ ]:
from pathlib import Path
import os


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for folder in [current, *current.parents]:
        if (folder / ".env").exists():
            return folder
    return current


def load_env_file(path: Path) -> list[str]:
    loaded_keys = []
    if not path.exists():
        return loaded_keys

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)
        os.environ.setdefault(key.upper(), value)
        loaded_keys.append(key)

    return loaded_keys


project_root = find_project_root()
loaded_keys = load_env_file(project_root / ".env")

api_key_names = [key for key in loaded_keys if key.lower().endswith("api_key")]

print(f"Loaded {len(loaded_keys)} key(s) from {project_root / '.env'}")
print("Loaded key name(s):", ", ".join(loaded_keys) if loaded_keys else "none")
print("API-like key present:", bool(api_key_names))

: 

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage


def summarize_messages(messages):
    return [
        {
            "class": message.__class__.__name__,
            "type": message.type,
            "content": message.content,
        }
        for message in messages
    ]

## Example 1: basic HumanMessage and AIMessage

In [ ]:
conversation = [
    HumanMessage(content="Suggest one feel-good movie for tonight."),
    AIMessage(content="Try The Intern. It is light, warm, and easy to watch."),
]

summarize_messages(conversation)

## Example 2: AIMessage with metadata

In [ ]:
answer = AIMessage(
    content="For a clever sci-fi thriller, watch Arrival.",
    response_metadata={"model": "demo-model", "finish_reason": "stop"},
    usage_metadata={"input_tokens": 18, "output_tokens": 11, "total_tokens": 29},
)

answer.model_dump()

## Example 3: AI tool call plus ToolMessage result

In [ ]:
tool_request = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "movie_lookup",
            "args": {"title": "Inception"},
            "id": "call_movie_1",
            "type": "tool_call",
        }
    ],
)

tool_result = ToolMessage(
    content='{"title": "Inception", "year": 2010, "rating": 8.8}',
    tool_call_id="call_movie_1",
    name="movie_lookup",
)

final_answer = AIMessage(
    content="Inception is a 2010 sci-fi action movie with an 8.8 rating."
)

summarize_messages([
    HumanMessage(content="Look up Inception."),
    tool_request,
    tool_result,
    final_answer,
])

## Optional: call a real chat model

In [ ]:
if os.getenv("GROQ_API_KEY"):
    try:
        from langchain_groq import ChatGroq
    except ModuleNotFoundError:
        print("Install langchain-groq to run this cell: uv add langchain-groq")
    else:
        llm = ChatGroq(model=os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"), temperature=0)
        response = llm.invoke([
            HumanMessage(content="Reply in one short sentence: what is ToolMessage used for?")
        ])
        response
elif os.getenv("OPENAI_API_KEY"):
    try:
        from langchain_openai import ChatOpenAI
    except ModuleNotFoundError:
        print("Install langchain-openai to run this cell: uv add langchain-openai")
    else:
        llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0)
        response = llm.invoke([
            HumanMessage(content="Reply in one short sentence: what is ToolMessage used for?")
        ])
        response
else:
    print("No supported API key found. Add groq_api_key or OPENAI_API_KEY to .env.")